##### Reprojecting and resampling R-factor raster downloaded from NOAA (https://data.noaa.gov/metaview/page?xml=NOAA/nos/ocm//iso/xml/48224.xml&view=getDataView&header=none). rasterio and rioxarray are important libraries for running the script

In [1]:
import rioxarray as rxr
from rasterio.enums import Resampling
from pyproj import Transformer

# --- paths
k_path   = "mart_rusle_output/K_US_kffact_0_25cm_UTM14_mart.tif"      # EPSG:26914 (10 m), your study-area raster
r_path   = "R-Factor-CONUS/R-Factor_CONUS.tif" # EPSG:5070 (800 m), national raster
out_path = "mart_rusle_output/R_factor_matched_to_K_mart.tif"

# --- open rasters (use chunks to limit memory; adjust chunk sizes as needed)
K = rxr.open_rasterio(k_path, masked=True)
R = rxr.open_rasterio(r_path, masked=True)

# ensure nodata is set (choose a value appropriate to your data)
if K.rio.nodata is None:
    K = K.rio.write_nodata(0, encoded=True)
if R.rio.nodata is None:
    R = R.rio.write_nodata(0, encoded=True)

# --- 1) compute K's bounding box in K's CRS
minx, miny, maxx, maxy = K.rio.bounds()  # EPSG:26914

# --- 2) transform that bbox into R's CRS (EPSG:5070) so we can clip R BEFORE reprojecting
transformer = Transformer.from_crs(K.rio.crs, R.rio.crs, always_xy=True)
minx5070, miny5070 = transformer.transform(minx, miny)
maxx5070, maxy5070 = transformer.transform(maxx, maxy)

# --- 3) clip R to that bbox in its own CRS -> BIG performance win
R_clipped = R.rio.clip_box(minx=minx5070, miny=miny5070, maxx=maxx5070, maxy=maxy5070)

# --- 4) reproject the clipped R to EXACTLY match K’s grid (CRS, res, transform, shape)
# R is continuous, so bilinear (or cubic) resampling is appropriate.
R_matched = R_clipped.rio.reproject_match(K, resampling=Resampling.bilinear)

# --- 5) write output
R_matched.rio.to_raster(out_path, compress="lzw", tiled=True)
print("Wrote:", out_path)

Wrote: mart_rusle_output/R_factor_matched_to_K_mart.tif
